# ChestScan AI - Classification de radiographies (version simple)

**But** : entrainer un reseau de neurones (CNN) TensorFlow/Keras pour dire si une radiographie thoracique est **NORMAL** ou **PNEUMONIA**, puis afficher ce modele dans une petite application Streamlit.

> **Prototype pedagogique. Ce modele n'est pas destine au diagnostic medical.**

## Le plan, en 6 etapes
1. **Les regles du jeu** : hyperparametres et graines aleatoires
2. **Les donnees** : compter, regarder, charger les images, separer train / validation / test
3. **Le modele** : un reseau simple, couche par couche
4. **L'entrainement** : fit, courbes, arret automatique
5. **L'evaluation** : uniquement sur le test, avec toutes les metriques
6. **La sauvegarde et l'application** : .keras, config.json, predict_xray, app.py

Dataset : [Kaggle - Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

## 1. Les regles du jeu

**Ce qu'on fait** : on declare EN HAUT du notebook tous les reglages (taille des images, nombre d'epochs, taille des lots, learning rate) et une graine aleatoire (SEED).

**Pourquoi** : c'est comme les regles d'un jeu avant de commencer. Si tout le monde utilise les memes regles, tout le monde peut refaire la meme experience. La SEED rend les resultats repetables : sans elle, chaque execution donne un modele legerement different.

In [ ]:
# Tous les reglages du projet sont ICI, en haut, pour qu on puisse tous les modifier vite
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    Input,
    MaxPooling2D,
    RandomFlip,
    RandomRotation,
    RandomZoom,
    Rescaling,
)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.utils import img_to_array, load_img

SEED = 42           # graine aleatoire : pour des resultats repetables
IMG_SIZE = 180      # taille des images (imposee par le sujet)
BATCH_SIZE = 32     # nombre d images envoyees ensemble a chaque pas
EPOCHS = 15         # nombre maximum d epochs (EarlyStopping arretera avant si besoin)
LEARNING_RATE = 1e-3  # pas d apprentissage de l optimiseur adam
SEUIL = 0.5         # si le score PNEUMONIA >= 0.5, on dit PNEUMONIA

# On fixe la graine partout : TensorFlow, NumPy et Python
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow", tf.__version__)
print("GPU utilise :", tf.config.list_physical_devices("GPU"))

## 2. Les donnees

**Ce qu'on fait** : on utilise le dataset Kaggle « chest-xray-pneumonia » : 5856 radiographies de poitrine au format JPEG, prises a des enfants de 1 a 5 ans dans un hopital de Guangzhou (Chine). Deux dossiers = deux classes : `NORMAL` et `PNEUMONIA`.

**Pourquoi** : le reseau n'apprend qu a partir de ce qu il voit. Connaître la provenance (quel age, quel hopital) est indispensable pour savoir a quoi croire ensuite : un modele entraine sur des enfants ne sera pas automatiquement juste sur des adultes.

In [ ]:
# Le meme code doit marcher en local ET sur Colab :
# - en local : dossier data/chest_xray (telecharge une fois, voir README.md)
# - sur Colab : on telecharge avec kagglehub (token Kaggle requis)
if os.path.exists(os.path.join("data", "chest_xray")):
    DATASET_PATH = os.path.join("data", "chest_xray")
elif os.path.exists("/kaggle/input/chest-xray-pneumonia/chest_xray"):
    DATASET_PATH = "/kaggle/input/chest-xray-pneumonia/chest_xray"
else:
    import kagglehub

    print("Telechargement du dataset via kagglehub (peut prendre plusieurs minutes)...")
    DATASET_PATH = os.path.join(
        kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"), "chest_xray"
    )

print("Dataset utilise :", DATASET_PATH)
print("Contenu :", sorted(os.listdir(DATASET_PATH)))

**Ce qu'on fait** : on compte les images de chaque dossier (train, val, test) et de chaque classe (NORMAL, PNEUMONIA).

**Pourquoi** : c est la toute premiere question a poser a un dataset. On découvre tout de suite le nombre d images ET le desequilibre (beaucoup plus de PNEUMONIA que de NORMAL).

In [ ]:
# On ne garde que les fichiers image : le dataset contient aussi des fichiers macOS
# (.DS_Store) qui ne sont pas des radiographies
EXTENSIONS_IMAGE = (".jpeg", ".jpg", ".png", ".bmp")
SPLITS = ["train", "val", "test"]
CLASSES = ["NORMAL", "PNEUMONIA"]

comptage = []
for dossier in SPLITS:
    for classe in CLASSES:
        chemin = os.path.join(DATASET_PATH, dossier, classe)
        fichiers = [f for f in os.listdir(chemin) if f.lower().endswith(EXTENSIONS_IMAGE)]
        comptage.append({"dossier": dossier, "classe": classe, "images": len(fichiers)})

df_comptage = pd.DataFrame(comptage)
tableau = df_comptage.pivot(index="dossier", columns="classe", values="images")
tableau["TOTAL"] = tableau.sum(axis=1)
print(tableau)
print("Total images sur le disque :", int(df_comptage["images"].sum()))

**Ce qu'on fait** : on dessine un graphique avec le nombre d images par classe, et on calcule le ratio.

**Pourquoi** : un chiffre seul se lit mal. Le graphique montre le desequilibre en un coup d oeil, et c est ce desequilibre qui va nous obliger a prendre des precautions plus loin (poids des classes, metriques par classe).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=df_comptage, x="dossier", y="images", hue="classe", ax=ax)
ax.set_title("Nombre d images par classe et par dossier")
ax.set_ylabel("Nombre d images")
plt.tight_layout()
plt.show()

# Le ratio se calcule sur le train : c est la que le modele apprend
n_normal = int(tableau.loc["train", "NORMAL"])
n_pneumo = int(tableau.loc["train", "PNEUMONIA"])
print(f"Train : {n_normal} NORMAL vs {n_pneumo} PNEUMONIA")
print(f"Ratio PNEUMONIA / NORMAL = {n_pneumo / n_normal:.2f}")

### Le desequilibre et la strategie (a dire a l oral)

Le train contient environ **2,9 fois plus de PNEUMONIA que de NORMAL**. Si on ne fait rien, le modele peut scorer haut en prevoyant surtout PNEUMONIA. Notre strategie, en 3 mesures :
1. **Poids des classes** : les images NORMAL peseent plus lourd dans la loss (calcul plus bas).
2. **Metriques par classe** : on regarde precision et recall de CHAQUE classe, pas seulement l accuracy globale.
3. **Pas de supprimage d images** : on ne jette aucune image, on corrige par les poids.

**Ce qu'on fait** : on regarde 4 images NORMAL et 4 images PNEUMONIA du train.

**Pourquoi** : avant d entrainer quoi que ce soit, on doit VORIR les donnees. Si les labels des dossiers ne correspondent pas aux vraies images, tout le travail est inutilisable. C est la regle : "on ne fait jamais confiance a un dataset sans l avoir regarde".

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ligne, classe in enumerate(CLASSES):
    dossier = os.path.join(DATASET_PATH, "train", classe)
    fichiers = sorted([f for f in os.listdir(dossier) if f.lower().endswith(EXTENSIONS_IMAGE)])
    for col in range(4):
        img = Image.open(os.path.join(dossier, fichiers[col * 10])).convert("RGB")
        axes[ligne, col].imshow(img)
        axes[ligne, col].set_title(classe)
        axes[ligne, col].axis("off")
plt.suptitle("Apercu des images (jeu d entrainement)")
plt.tight_layout()
plt.show()

### Chargement des images en memoire

**Ce qu'on fait** : une simple boucle `for`. Pour chaque fichier : on l ouvre avec PIL, on le convertit en RGB (3 canaux), on le redimensionne en 180x180, et on le met dans un tableau NumPy **en uint8** (entiers de 0 a 255). Une image cassee leve une erreur, attrapee par `try/except` : elle est comptee et excludee.

**Pourquoi** :
- **uint8** = 3 fois moins de memoire que des decimaux (1 octet par pixel au lieu de 4). Le modele fera la division par 255 lui-meme (couche Rescaling).
- **try/except** = un fichier corrompu ne doit pas faire planter tout le chargement.
- **Les labels viennent des noms de dossiers** : NORMAL = 0, PNEUMONIA = 1.

In [ ]:
# On charge TOUTES les images en memoire, une par une.
# Resultat : 2 tableaux NumPy. Les images restent en uint8 (0 a 255), sans division.
x_images = []   # les pixels
y_labels = []   # les labels : 0 = NORMAL, 1 = PNEUMONIA
chemins = []    # le chemin de chaque image (utile pour plus tard)
corrompues = [] # les fichiers illisibles, pour les lister

for dossier in SPLITS:
    for classe in CLASSES:
        label = CLASSES.index(classe)  # le label vient du NOM du dossier
        dossier_complet = os.path.join(DATASET_PATH, dossier, classe)
        for fichier in sorted(os.listdir(dossier_complet)):
            if not fichier.lower().endswith(EXTENSIONS_IMAGE):
                continue  # on ignore .DS_Store et autres fichiers non image
            chemin = os.path.join(dossier_complet, fichier)
            try:
                img = Image.open(chemin).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                x_images.append(np.array(img, dtype=np.uint8))  # uint8 : 0 a 255
                y_labels.append(label)
                chemins.append(chemin)
            except Exception:
                # Le fichier est casse : on le note et on continue
                corrompues.append(chemin)

x_images = np.array(x_images, dtype=np.uint8)
y_labels = np.array(y_labels, dtype=np.uint8)

print("Images chargees :", len(x_images))
print("Images corrompues exclues :", len(corrompues))
for chemin in corrompues:
    print("  -", chemin)
print("Forme du tableau images :", x_images.shape)

**Ce qu'on fait** : on verifie une image chargee (sa taille, sa forme, la plage de ses valeurs).

**Pourquoi** : une verification bete mais efficace. `shape` doit etre `(180, 180, 3)` et les valeurs doivent rester entre 0 et 255. Si ce n est pas le cas, quelque chose ne va pas dans le chargement.

In [ ]:
print("Forme d une image :", x_images.shape[1:], "-> (hauteur, largeur, canaux RGB)")
print("Valeur minimale :", int(x_images.min()), "| maximale :", int(x_images.max()), "(0 a 255, pas encore normalise)")

# On re-affiche une image du tableau charge : si elle s affiche, le chargement est bon
plt.figure(figsize=(4, 4))
plt.imshow(x_images[0])
plt.title("Une image chargee en memoire")
plt.axis("off")
plt.show()

### Separation train / validation / test

**Ce qu'on fait** : on separe les donnees en 3 morceaux, **avec deux appels a `train_test_split`** :
1. d abord on reserve le **TEST** (20 %) : il deviendra independant, on n y touchera plus ;
2. ensuite, dans le reste, on reserve la **VALIDATION** (20 %).

**Pourquoi** :
- Le **test independant** sert UNIQUEMENT a la fin, pour juger le modele. S on s en servait plus tot, ce serait comme corriger la copie avec la correction sous les yeux (triche, appelee "fuite de donnees").
- La **validation** sert a l EarlyStopping : elle dit au modele "arrete, tu commences a memoriser".
- **`stratify`** garde la meme proportion de classes dans chaque morceau.
- Le dossier `val` du dataset ne contient que 16 images : il est inutilisable, on le melange au train.

In [ ]:
# On garde les chemins et labels pour savoir QUI est dans quel morceau
df_total = pd.DataFrame({"chemin": chemins, "label": y_labels, "classe": [CLASSES[int(l)] for l in y_labels]})

# Etape 1 : on reserve le test (20 %), independant de tout le reste
df_hors_test, df_test = train_test_split(
    df_total, test_size=0.2, stratify=df_total["classe"], random_state=SEED
)

# Etape 2 : dans le reste, on reserve la validation (20 % du total)
df_train, df_val = train_test_split(
    df_hors_test, test_size=0.2, stratify=df_hors_test["classe"], random_state=SEED
)

print(f"Train      : {len(df_train)} images")
print(f"Validation : {len(df_val)} images")
print(f"Test       : {len(df_test)} images  <- independant, utilise UNE seule fois a la fin")

# Les tableaux de pixels sont tries dans l ordre du chargement : on les reprend
# avec les index conservees par le split, pour que chaque label reste colle a sa bonne image
index_train = df_train.index.to_numpy()
index_val = df_val.index.to_numpy()
index_test = df_test.index.to_numpy()

x_train, y_train = x_images[index_train], y_labels[index_train]
x_val, y_val = x_images[index_val], y_labels[index_val]
x_test, y_test = x_images[index_test], y_labels[index_test]

# On verifie que la proportion de classes est la meme partout (~23 % NORMAL)
for nom, y_part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    part_normal = 100.0 * (y_part == 0).mean()
    print(f"{nom:11s} => NORMAL {part_normal:5.1f} % | PNEUMONIA {100 - part_normal:5.1f} %")

**A retenir pour l'oral** : « Le test est mis de cote DES LE DEBUT, avant toute normalisation, et il ne sera ouvert qu une seule fois, a la fin. La validation sert a arreter l entrainement, jamais a juger le modele final. »

### Le poids des classes (correction du desequilibre)

**Ce qu'on fait** : on calcule un poids pour chaque classe avec une formule simple, visible :

`poids_classe = total_des_images / (2 * nombre_d_images_de_la_classe)`

**Pourquoi** : la classe la plus rare (NORMAL) recevra un poids superieur a 1, la classe majoritaire inferieur a 1. Pendant l entrainement, une image NORMAL ratee coutera donc plus cher qu une image PNEUMONIA ratee : le modele sera oblige de les apprendre serieusement, sans qu on jette d images.

In [ ]:
# Formule simple et symetrique : poids = total / (2 * nombre de la classe)
total_train = len(y_train)
nombre_normal = int((y_train == 0).sum())
nombre_pneumo = int((y_train == 1).sum())

poids_normal = total_train / (2 * nombre_normal)
poids_pneumo = total_train / (2 * nombre_pneumo)
CLASS_WEIGHT = {0: poids_normal, 1: poids_pneumo}

print(f"Train : {nombre_normal} NORMAL, {nombre_pneumo} PNEUMONIA, total {total_train}")
print(f"Poids NORMAL   = {total_train} / (2 x {nombre_normal}) = {poids_normal:.3f}")
print(f"Poids PNEUMONIA = {total_train} / (2 x {nombre_pneumo}) = {poids_pneumo:.3f}")
print("Envoye a model.fit() :", CLASS_WEIGHT)

## 3. Le modele

**Ce qu'on fait** : on empile des couches dans un `Sequential` (de gauche a droite, comme une chaene).

| Couche | Role, en une phrase |
|--------|--------------------|
| `Rescaling(1/255)` | divise les pixels par 255 : **c est la normalisation, elle vit dans le modele** |
| `RandomFlip / Rotation / Zoom` | deformations legeres, **uniquement a l entrainement** (comme des exercices d echole) |
| `Conv2D` | cherche des formes (bords, taches claires) avec de petits filtres |
| `MaxPooling2D` | reduit la taille de la carte en gardant l information utile |
| `Flatten` | aplatis le tableau 2D en un seul grand vecteur |
| `Dense(128, relu)` | couche de decision classique |
| `Dropout(0.5)` | eteint 50 % des neurones au hasard : evite la memorisation (overfitting) |
| `Dense(1, sigmoide)` | **sortie binaire** : une probabilite entre 0 et 1 |

**Le point clé a dire a l oral** : la normalisation et l augmentation sont DANS le modele. Du coup, l application Streamlit ne peut pas les oublier : elle envoie des pixels de 0 a 255, et le modele fait tout lui-meme.

In [ ]:
# Le reseau, couche par couche. Chaque couche a un commentaire qui dit POURQUOI elle est la.
model = Sequential(
    [
        Input(shape=(IMG_SIZE, IMG_SIZE, 3)),  # on dit la forme d entree : 180x180 en couleur

        # --- Pretraitement et augmentation : DANS le modele ---
        # Rescaling divise par 255 : c est LA normalisation, en un seul endroit du projet
        Rescaling(1.0 / 255),
        # Ces 3 couches ne travaillent QUE pendant l entrainement (training=True).
        # A la prediction, elles deviennent de simples passages a vide.
        RandomFlip("horizontal"),   # retournement horizontal aleatoire
        RandomRotation(0.1),        # rotation aleatoire de 10 % au maximum
        RandomZoom(0.1),            # zoom aleatoire de 10 % au maximum

        # --- Extraction de caracteristiques : 3 blocs Conv + Pool ---
        # Conv2D = filtres qui detectent des formes (bords, taches claires)
        Conv2D(32, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),  # on divise la taille par 2, on garde l essentiel
        Conv2D(64, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),

        # --- Classification ---
        Flatten(),                              # on aplatis tout en un vecteur
        Dense(128, activation="relu"),          # couche de decision
        Dropout(0.5),                           # on eteint 50 % des neurones au hasard
        # Sortie binaire : 1 seul neurone + sigmoide = probabilite d etre PNEUMONIA
        Dense(1, activation="sigmoid"),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",  # la loss des 2 classes opposees
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)
model.summary()

**A retenir pour l'oral** : « Le pretraitement est dans le modele : la couche Rescaling divise par 255, et les couches d augmentation ne s activent qu a l entrainement. C est pourquoi l application n a A FAIRE au pretraitement : le modele le fait lui-meme. »

## 4. L'entrainement

**Ce qu'on fait** : on appelle `model.fit()` avec deux "arbitres" :
- **EarlyStopping** : si la validation ne s ameliore plus pendant 4 epochs, on arrete et on rend les MEILLEURS poids. C est le remede principal au surapprentissage (overfitting = memoriser le train au lieu d apprendre des regles generales).
- **ReduceLROnPlateau** : si la validation stagne, on divise le learning rate par 2. Le modele peut alors avancer plus finement.

On mesure aussi la **duree totale** avec `time.time()`.

In [ ]:
# Arret automatique : on arrete si la validation ne progresse plus (evite de memoriser)
early_stop = EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
)
# Si la validation stagne, on baisse le learning rate (le modele avance plus finement)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

debut = time.time()
history = model.fit(
    x_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    class_weight=CLASS_WEIGHT,  # correction du desequilibre
    callbacks=[early_stop, reduce_lr],
)
duree = time.time() - debut
print(f"Duree de l entrainement : {duree / 60:.1f} minutes")

**Ce qu'on fait** : on dessine les courbes d entrainement : la loss et l accuracy, en train et en validation, epoch par epoch.

**Pourquoi** : c est ici qu on voit si le modele apprend vraiment ou s il commence a memoriser.
- **Surapprentissage (overfitting)** : la courbe train continue de baisser, la courbe validation remonte.
- **Sous-apprentissage** : les deux courbes restent mauvaises.
- Un ecart faible entre les deux courbes = le modele generalise bien.

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
gamme = range(1, len(acc) + 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(gamme, acc, "o-", label="Train")
ax[0].plot(gamme, val_acc, "o-", label="Validation")
ax[0].set_title("Accuracy : on regarde l ecart entre les deux courbes")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(gamme, loss, "o-", label="Train")
ax[1].plot(gamme, val_loss, "o-", label="Validation")
ax[1].set_title("Loss (plus bas = mieux)")
ax[1].set_xlabel("Epoch")
ax[1].legend()
plt.tight_layout()
plt.show()

print("Epochs executes :", len(acc), "/", EPOCHS)
print("Meilleure val_accuracy :", round(max(val_acc), 4))
print("Meilleure val_loss     :", round(min(val_loss), 4))
print("Ecart final train / val (accuracy) :", round(abs(acc[-1] - val_acc[-1]), 4))

**A retenir pour l'oral** : « Je regarde l ecart entre la courbe train et la courbe validation. Si l ecart grandit, c est du surapprentissage : j ai alors le Dropout, l augmentation, les poids des classes et l EarlyStopping comme remedes. »

## 5. L'evaluation sur le test independant

**Ce qu'on fait** : une fois, et seulement une fois, on ouvre le test. On y calcule toutes les metriques.

**Pourquoi** : le test est le seul morceau que le modele n a JAMAIS vu. C est lui qui donne une estimation honnete de la performance reelle. Les autres metriques (train, validation) sont biaisees : le modele a deja repondu dessus pendant l'entrainement.

Lecture des metriques :
- **accuracy** : proportion globale de bonnes reponses (trompeuse avec le desequilibre) ;
- **precision** : parmi les predits PNEUMONIA, combien le sont vraiment ;
- **recall** : parmi les vrais PNEUMONIA, combien ont ete detects ;
- **F1** : moyenne harmonique des deux ;
- **ROC-AUC** : capacite a separer les deux classes.

In [ ]:
# Une seule evaluation, sur le test independant
y_pred_proba = model.predict(x_test).ravel()     # probabilite d etre PNEUMONIA
y_pred = (y_pred_proba >= SEUIL).astype(int)     # decision au seuil 0.5

accuracy_test = float((y_pred == y_test).mean())
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy  : {accuracy_test:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=CLASSES))

**Ce qu'on fait** : on dessine la matrice de confusion et on compte les erreurs une par une : vrais negatifs, faux positifs, faux negatifs, vrais positifs.

**Pourquoi** : la matrice est la lecture CONCRETE des erreurs.
- **Faux negatif** : un malade predit NORMAL. **Le plus grave en medical** : ce patient repart sans traitement.
- **Faux positif** : un sain predit malade. C est un examen supplementaire inutile et du stress, mais pas de danger direct.

In [ ]:
matrice = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matrice, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
)
ax.set_xlabel("Classe predite")
ax.set_ylabel("Vraie classe")
ax.set_title("Matrice de confusion (test independant)")
plt.show()

vrais_negatifs = int(matrice[0, 0])  # NORMAL predit NORMAL
faux_positifs = int(matrice[0, 1])   # NORMAL predit PNEUMONIA
faux_negatifs = int(matrice[1, 0])   # PNEUMONIA predit NORMAL  <- le plus grave
vrais_positifs = int(matrice[1, 1])  # PNEUMONIA predit PNEUMONIA

print("Vrais negatifs (sains bien identifies)   :", vrais_negatifs)
print("Faux positifs (sains pris pour malades)  :", faux_positifs)
print("Vrais positifs (malades bien identifies) :", vrais_positifs)
print("Faux negatifs (malades rates)            :", faux_negatifs)

**A retenir pour l'oral** : « En depistage, un faux negatif laisse un malade sans traitement, donc je regarde d abord le recall de la classe PNEUMONIA. C est aussi pour ca que le seuil de decision est un arbitrage medical, pas technique. »

**Ce qu'on fait** : on trace la courbe ROC, puis on recalcule precision et recall pour differents seuils de decision (0.3, 0.4, 0.5, 0.6, 0.7).

**Pourquoi** : baisser le seuil (par ex. a 0.3) augmente le recall : moins de malades rates, mais plus de fausses alarmes. On voit le compromis dans un tableau, au lieu de le subir.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
ax[0].plot([0, 1], [0, 1], "k--", label="hasard (AUC = 0.5)")
ax[0].set_xlabel("Taux de faux positifs")
ax[0].set_ylabel("Taux de vrais positifs (recall)")
ax[0].set_title("Courbe ROC (test independant)")
ax[0].legend()

lignes = []
for seuil_test in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_seuil = (y_pred_proba >= seuil_test).astype(int)
    lignes.append(
        {
            "seuil": seuil_test,
            "precision": precision_score(y_test, pred_seuil, zero_division=0),
            "recall": recall_score(y_test, pred_seuil, zero_division=0),
            "f1": f1_score(y_test, pred_seuil, zero_division=0),
        }
    )
df_seuils = pd.DataFrame(lignes).set_index("seuil")
print(df_seuils.round(3))

ax[1].plot(df_seuils.index, df_seuils["precision"], "o-", label="precision")
ax[1].plot(df_seuils.index, df_seuils["recall"], "o-", label="recall")
ax[1].plot(df_seuils.index, df_seuils["f1"], "o-", label="f1")
ax[1].set_xlabel("Seuil de decision")
ax[1].set_title("Precision / recall / f1 selon le seuil")
ax[1].legend()
plt.tight_layout()
plt.show()

**Ce qu'on fait** : on affiche 6 predictions correctes et 6 predictions incorrectes, avec le score pour chacune.

**Pourquoi** : voir les erreurs les plus concretes. Souvent, les fausses alarmes sont des cas limites (score proche de 0.5), et les faux negatifs sont des images moins typiques ou floues.

In [ ]:
index_corrects = np.where(y_pred == y_test)[0]
index_incorrects = np.where(y_pred != y_test)[0]


def afficher_exemples(indexes, titre, n_max=6):
    n = min(n_max, len(indexes))
    if n == 0:
        print(titre, ": aucun exemple")
        return
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for i in range(n):
        idx = int(indexes[i])
        ligne, col = divmod(i, 3)
        axes[ligne, col].imshow(x_test[idx])
        reel = CLASSES[int(y_test[idx])]
        dit = CLASSES[int(y_pred[idx])]
        axes[ligne, col].set_title(f"Vrai : {reel} | Predit : {dit}\nscore = {y_pred_proba[idx]:.2f}")
        axes[ligne, col].axis("off")
    plt.suptitle(titre)
    plt.tight_layout()
    plt.show()


afficher_exemples(index_corrects, "Predictions correctes")
# On montre d abord les faux negatifs (les plus critiques), puis les faux positifs
index_fn = [i for i in index_incorrects if y_test[i] == 1]
index_fp = [i for i in index_incorrects if y_test[i] == 0]
afficher_exemples(index_fn, "Faux negatifs (malade rate)")
afficher_exemples(index_fp, "Faux positifs (fausse alarme)")
print(f"Erreurs sur le test : {len(index_incorrects)} / {len(y_test)}")

**Ce qu'on fait** : on compare les trois jeux avec la MEME evaluation (`model.evaluate`), sans augmentation.

**Pourquoi** : les trois scores doivent etre proches.
- **train >> test** : surapprentissage (overfitting), le modele a memorise.
- **les trois faibles** : sous-apprentissage, le modele n a pas appris grand chose.
- attention : l evaluation du train est faite ici SANS augmentation (les couches Random sont desactivees hors entrainement), ce qui rend la comparaison honnete.

In [ ]:
# model.evaluate utilise training=False : les couches d augmentation sont donc inactives
resultats_train = model.evaluate(x_train, y_train, verbose=0)
resultats_val = model.evaluate(x_val, y_val, verbose=0)
resultats_test = model.evaluate(x_test, y_test, verbose=0)

df_comparaison = pd.DataFrame(
    {"train": resultats_train, "validation": resultats_val, "test": resultats_test},
    index=model.metrics_names,
)
print(df_comparaison.round(4))

### Analyse critique des resultats

Questions a se poser en regardant les chiffres au-dessus :
1. **Quel est l ecart train / validation / test ?** S il est petit, le modele generalise.
2. **Ou sont les erreurs ?** Avec le desequilibre, la majorite des erreurs est en faux positifs (NORMAL pris pour PNEUMONIA).
3. **Quelle erreur est grave ?** En depistage, le **faux negatif** (malade rate) l est bien plus que le faux positif. C est pourquoi on surveille le **recall** et on ajuste le **seuil** : un seuil a 0.3 recoit plus de malades, au prix de fausses alarmes.
4. **Le score est il une probabilite reelle ?** Non : c est la confiance du modele. Un score de 0.9 ne veut pas dire "90 % malade".

## 6. La sauvegarde et l'utilisation

**Ce qu'on fait** : on sauvegarde deux fichiers :
- `chestscan_model.keras` : le modele entraine, **avec sa couche Rescaling incluse** ;
- `config.json` : les informations dont l application a besoin (taille d image, noms des classes, seuil).

**Pourquoi** : l application n a jamais besoin de re-entrainer. Elle charge ces deux fichiers et c est tout. C est ce qui rend le deploiement possible.

In [ ]:
model.save("chestscan_model.keras")

# config.json = ce que l application doit savoir SANS relire le notebook
config = {
    "image_size": [IMG_SIZE, IMG_SIZE],
    "classes": CLASSES,  # index 0 = NORMAL, index 1 = PNEUMONIA
    "threshold": SEUIL,
    "note": "Le modele contient sa propre couche Rescaling(1/255) : l application envoie des pixels 0-255.",
    "dataset": "Kaggle paultimothymooney/chest-xray-pneumonia",
    "resultats_test": {
        "accuracy": round(accuracy_test, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "roc_auc": round(roc_auc, 4),
    },
}
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Fichiers ecrits : chestscan_model.keras et config.json")
print(json.dumps(config["resultats_test"], indent=2))

**Ce qu'on fait** : on recharge le `.keras` depuis le disque et on recalcule les predictions du test. On verifie que ce sont exactement les memes qu avant.

**Pourquoi** : c est le test de deploiement. Si le modele recharge ne redonne pas les memes chiffres, quelque chose ne va pas dans la sauvegarde.

In [ ]:
model_recharge = load_model("chestscan_model.keras")
y_proba_recharge = model_recharge.predict(x_test).ravel()

identique = np.allclose(y_proba_recharge, y_pred_proba, atol=1e-6)
print("Predictions identiques apres rechargement :", bool(identique))
print("Plus grand ecart entre les scores :", float(np.max(np.abs(y_proba_recharge - y_pred_proba))))

### La fonction `predict_xray(image_path)`

**Ce qu'on fait** : une fonction qui charge une image du disque, la prepare et la fait passer dans le modele. Elle retourne la classe predite et le score PNEUMONIA.

**Le point essentiel** : on convertit en RGB et on redimensionne en 180x180, mais on **NE DIVISE PAS par 255**. Pourquoi ? Parce que la couche `Rescaling` est dans le modele : elle fait la division elle-meme. Si on divisait ici, ce serait une double normalisation, et les predictions seraient fausses.

In [ ]:
def predict_xray(image_path):
    # Charge une image, la prepare, et predit. Retourne (classe predite, score PNEUMONIA).
    # Meme preparation que l entrainement : RGB + 180x180
    # PAS de division par 255 ici : la couche Rescaling du modele le fait deja
    img = Image.open(image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    x = np.array(img, dtype=np.uint8)
    x = np.expand_dims(x, axis=0)  # le modele attend un lot, meme pour une seule image

    score = float(model_recharge.predict(x, verbose=0)[0][0])  # probabilite PNEUMONIA
    classe = CLASSES[1] if score >= SEUIL else CLASSES[0]
    return classe, score

**Ce qu'on fait** : on teste `predict_xray` sur 6 images prises dans le test independant (3 NORMAL et 3 PNEUMONIA), puis sur une image qui n existe pas (pour montrer que la fonction echoue proprement).

**Pourquoi** : une fonction non testee est une fonction dont on ne sait rien. On veut voir la vraie classe, la classe predite, et le score.

In [ ]:
exemples = pd.concat(
    [
        df_test[df_test["classe"] == "NORMAL"].head(3),
        df_test[df_test["classe"] == "PNEUMONIA"].head(3),
    ]
)

lignes_test = []
for _, rang in exemples.iterrows():
    classe, score = predict_xray(rang["chemin"])
    lignes_test.append(
        {
            "image": os.path.basename(rang["chemin"]),
            "vraie_classe": rang["classe"],
            "classe_predite": classe,
            "score_pneumonia": round(score, 3),
            "juste": classe == rang["classe"],
        }
    )
df_test_predict = pd.DataFrame(lignes_test)
print(df_test_predict.to_string(index=False))
print(f"Bonnes predictions sur cet echantillon : {df_test_predict['juste'].sum()} / {len(df_test_predict)}")

In [ ]:
# Test d une entree invalide : la fonction doit echouer proprement
# (l application Streamlit affiche alors un message clair au lieu de planter)
try:
    predict_xray("image_inexistante.jpg")
except Exception as erreur:
    print("Entree invalide geree :", type(erreur).__name__)

## Limites et pistes d'amelioration

### Limites (a connaitre pour l oral)
- **Population specifique** : des enfants de 1 a 5 ans, un seul hopital. Le score du test ne garantit rien sur des adultes ni sur un autre hopital.
- **Test non externe** : train et test viennent du meme hopital et de la meme periode.
- **Binaire simpliste** : la pneumonie a des causes tres differentes (bacterienne, virale) ; le modele reconnait une apparence, pas une cause.
- **Score non calibre** : la sigmoide donne la confiance du modele, pas une probabilite medicale.
- **Boite noire** : le modele ne dit pas quelles zones de l image ont fait pencher la decision.

### Pistes d'amelioration (hors flux principal, non demandees)
1. **Transfer learning** (VGG16, EfficientNetB0 pre-entraines) : plus performant, mais moins lisible a expliquer.
2. **Grad-CAM** : afficher les zones de l image qui ont decide.
3. **Seuil metier** : choisir le seuil selon le cout des faux negatifs (voir le tableau des seuils).
4. **Donnees externes** (NIH, MIMIC-CXR) : mesurer la vraie degradation du modele.

---

### A retenir, en 4 phrases
- **Pas de fuite de donnees** : le test est mis de cote des le debut, et rien ne s en sert avant la fin.
- **Un seul pretraitement** : il est DANS le modele (Rescaling), donc l application ne peut pas l oublier.
- **En medical, le recall compte** : un faux negatif laisse un malade sans traitement.
- **Le seuil est un arbitrage medical** : on peut le changer sans reentrainer le modele.